# Splatial on Colab GPU — high-view reconstruction

Our local box is a 12 GB RTX 4070 Ti, which caps AnySplat at ~16 views / 448×616. A Colab **A100 (40 GB)** or **L4 (24 GB)** lifts that cap, so we can test the two things 12 GB couldn't:
1. **Feed-forward at 30–60+ views and higher resolution** → denser room surfaces.
2. **Post-opt with many views** → where it stops overfitting and actually helps.

**Runtime → Change runtime type → GPU (A100 or L4 preferred).** Then run the cells top to bottom.

Outputs a `scene.ply` + `scene_mobile.ply` you download and drop into `web/scenes/<id>/` locally to view in the Three.js viewer.

## 1. Check the GPU you got

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# A100 40GB / L4 24GB = lots of headroom (use 40-64 views). T4 16GB = modest (use ~24). Free tier may give T4.

## 2. Clone AnySplat + our pipeline, install deps
Takes ~5–10 min (torch + AnySplat requirements). The symlink makes our CLI and `postopt.sh` find AnySplat.

In [ ]:
%cd /content
![ -d AnySplat ] || git clone https://github.com/InternRobotics/AnySplat.git
# Our pipeline (rotation-correct frame sampling, tall crop, PLY export, held-out eval).
# If the repo is private, replace with your token URL or upload a zip instead.
![ -d splatial ] || git clone https://github.com/xji6xu4m3/splatial.git
!ln -sfn /content/AnySplat /content/splatial/external_AnySplat

# Torch pinned to AnySplat's version, then its requirements, then our extras.
!pip install -q torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121
%cd /content/AnySplat
!pip install -q -r requirements.txt
!pip install -q open3d plyfile lpips scikit-image opencv-python-headless huggingface_hub
print('\n✓ install done')

## 3. Upload a capture video
Use one of the originals from `videos/` (e.g. `room1.MOV`). Portrait `.MOV` is fine — our frame extractor honors the rotation flag.

In [ ]:
from google.colab import files
up = files.upload()                      # pick room1.MOV / pet1.MOV
VIDEO = '/content/' + next(iter(up))
print('uploaded:', VIDEO)

## 4. Reconstruct at high view count + measure held-out PSNR
On 12 GB our ceiling was 16 views @ 616. Here push it. **Tune `MAX_VIEWS` / `CROP_LONG_CAP` to the GPU** — A100/L4 can take 40–64 views and the full 784 crop. The CLI has an OOM-recovery ladder, so it degrades instead of crashing.

In [ ]:
import os
os.environ['ANYSPLAT_ROOT'] = '/content/AnySplat'
os.environ['RECON_ENGINE']  = 'anysplat'
os.environ['MAX_VIEWS']     = '48'      # 12GB did 16; A100/L4 try 40-64
os.environ['MIN_VIEWS']     = '24'
os.environ['CROP_LONG_CAP'] = '784'     # full portrait FOV (12GB had to use 616)
os.environ['CAPTURE_RATE']  = '2.0'     # more frames/sec so the sampler can reach the higher cap
os.environ['SCENE_MODE']    = 'room'    # 'object' for a single-subject scan (floater cleanup)
SCENE = 'hires'
%cd /content/splatial
!python -m modules.reconstruct.cli "$VIDEO" scenes $SCENE
print('\n--- held-out PSNR/SSIM/LPIPS (compare to local 12GB baseline: room1 17.55) ---')
!python experiments/eval_heldout.py --scene $SCENE --holdout 6 --tag colab_hi

## 5. (Optional) Post-opt with many views — the real test
On 12 GB post-opt overfit at ≤17 views. With 40–100 views it should finally *help*. Needs `fused_ssim` (CUDA build — a few min). Trains then writes a refined PLY; `postopt_to_scene.py` makes it viewer-ready (SH0 / opacity-logit / needle cleanup).

In [ ]:
!pip install -q git+https://github.com/rahul-goel/fused-ssim.git
!pip install -q tyro tensorboard viser nerfview splines
%cd /content/splatial
# postopt.sh runs AnySplat's gsplat trainer; SH0 is forced inside it for our viewer.
!bash tools/postopt.sh "/content/splatial/scenes/$SCENE/frames" "/content/splatial/scenes/$SCENE/postopt" 3000
!python tools/postopt_to_scene.py "scenes/$SCENE/postopt/ply/"point_cloud_*.ply ${SCENE}_po --base-scene scenes/$SCENE/scene.json
print('refined held-out metrics: scenes/'+SCENE+'/postopt/stats/val_step*.json')

## 6. Download the scene to view locally
Unzip into `web/scenes/<id>/` on your machine, then open `http://localhost:5173/?scene=<id>`.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive(f'/content/{SCENE}', 'zip', f'/content/splatial/scenes/{SCENE}')
files.download(f'/content/{SCENE}.zip')